# MOP 3231 · Task 3 — Midterm Project: Will the Client Subscribe? (Bank Marketing)

**Student:** Imamsidikov · **Data:** `data/train.csv` (32,940 labelled calls) and `data/test.csv` (8,236 calls to score),
derived from UCI Bank Marketing (`bank-additional-full.csv`, CC BY 4.0)

| Part | Topic |
|---|---|
| 0 | Setup, versions, SEED |
| 1 | Loading and checking the data |
| 2 | EDA: the target and the numeric features |
| 3 | EDA: the categorical features, `unknown`, `pdays = 999` and `duration` |
| 4 | Validation split and the preprocessing pipeline |
| 5 | Baselines and stronger models |
| 6 | Cross-validation and tuning |
| 7 | Final evaluation and threshold |
| 8 | Refit, save, reload and score `test.csv` |

This is a teaching dataset, not evidence about real banks or clients.

## Part 0 · Setup

Virtual environment `.Imamsidikov_ML_task_3` (Python 3.12) with the pinned `requirements.txt`; Jupyter is only the tool
and stays out of it.

**SEED = 2954**: the last four digits of my student ID (S24072954). It seeds every split, fold and model.

In [1]:
import sys

import joblib
import matplotlib
import numpy
import pandas
import scipy
import sklearn

print(sys.version.split()[0], numpy.__version__, scipy.__version__, pandas.__version__,
      matplotlib.__version__, sklearn.__version__, joblib.__version__)

3.12.12 2.3.4 1.16.3 2.3.3 3.10.7 1.7.2 1.5.2


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 2954
TARGET = "y"

# Chart style: recessive grid and axes, one colour per class used the same way in every figure
NO, YES, NEUTRAL = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e",
    "axes.titlesize": 11,
    "axes.grid": True,
    "grid.color": "#e7e6e1",
    "grid.linewidth": 0.8,
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "legend.frameon": False,
})

## Part 1 · Loading and checking the data

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print("train:", train.shape, "| test:", test.shape)
print("in train only:", sorted(set(train.columns) - set(test.columns)))
print("in test only: ", sorted(set(test.columns) - set(train.columns)))

train: (32940, 22) | test: (8236, 20)
in train only: ['duration', 'y']
in test only:  []


In [4]:
# Column types and the values seen in train
summary = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(),
    "missing": train.isna().sum(),
    "example": train.iloc[0],
})
summary

,dtype,n_unique,missing,example
id,int64,32940,0,0
age,int64,76,0,39
job,object,12,0,management
marital,object,4,0,single
education,object,8,0,basic.9y
default,object,3,0,unknown
housing,object,3,0,yes
loan,object,3,0,no
contact,object,2,0,telephone
month,object,10,0,may


In [5]:
checks = {
    "missing cells in train": int(train.isna().sum().sum()),
    "missing cells in test": int(test.isna().sum().sum()),
    "train ids unique": train["id"].is_unique,
    "test ids unique": test["id"].is_unique,
    "train and test ids disjoint": set(train["id"]).isdisjoint(test["id"]),
    "duplicate rows in train (ignoring id)": int(train.drop(columns="id").duplicated().sum()),
    "target values": sorted(train[TARGET].unique()),
}
for name, value in checks.items():
    print(f"{name:40s} {value}")

missing cells in train                   0
missing cells in test                    0
train ids unique                         True
test ids unique                          True
train and test ids disjoint              True
duplicate rows in train (ignoring id)    0
target values                            ['no', 'yes']


In [6]:
# Same categories in train and test? A level seen only in test would need handle_unknown in the encoder.
categorical = [c for c in test.columns if test[c].dtype == object]
numeric = [c for c in test.columns if c not in categorical and c != "id"]
for c in categorical:
    only_test = set(test[c]) - set(train[c])
    if only_test:
        print(c, "levels only in test:", only_test)
print(len(categorical), "categorical:", categorical)
print(len(numeric), "numeric:", numeric)

10 categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
9 numeric: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


**What the checks say**

- No missing cells (`NaN`) in either file, unique and disjoint `id`s, no duplicate rows. Missing information is coded
  as values instead: the string `unknown` in six categorical columns and `pdays = 999` (Part 3).
- `train.csv` has two columns `test.csv` lacks: the target `y` and `duration`. A model can only use what `test.csv`
  has, so the feature set is the 19 columns of `test.csv` minus `id`: 10 categorical and 9 numeric.
- `id` is a shuffled row number added when the files were made. It carries no information and is not a feature.